# TUGAS MANDIRI PERTEMUAN 12 — PRAKTIKUM BIG DATA
## Klasifikasi & ML Pipeline dengan Spark MLlib (Studi Kasus: Kopi Nusantara)

| | |
|---|---|
| **Nama Lengkap** | Muhamad Dimas Adji Saputra |
| **NPM** | 2505060078 |
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Pertemuan** | 12 |

---
## Persiapan: Inisialisasi SparkSession & Pembuatan Dataset

In [1]:
import os
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Inisialisasi SparkSession (Spark/PySpark 3.5.9)
spark = SparkSession.builder \
    .appName("Tugas12-KopiNusantara-MLlib") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

26/11/25 20:03:43 WARN Utils: Your hostname, Dimsx resolves to a loopback address: 127.0.1.1; using 192.168.100.21 instead (on interface wlp4s0)
26/11/25 20:03:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 20:03:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
# 2. Menyiapkan Dataset tugas12
if not os.path.exists("tugas12_ulasan.csv"):
    np.random.seed(212)
    n = 3000
    menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
    cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
    waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

    data_tugas12 = pd.DataFrame({
        "nama_menu": np.random.choice(menu_list, size=n),
        "cabang": np.random.choice(cabang_list, size=n),
        "waktu_pesan": np.random.choice(waktu_list, size=n),
        "qty": np.random.randint(1, 6, size=n),
        "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
    })

    # Simulasi: waktu tunggu lebih lama cenderung menurunkan peluang ulasan positif
    prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
    data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)
    data_tugas12.to_csv("tugas12_ulasan.csv", index=False)

# Membaca dataset CSV ke dalam Spark DataFrame
df = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
print(f"Total baris dataset: {df.count()}")
df.show(5)
print("Distribusi label (ulasan_positif):")
df.groupBy("ulasan_positif").count().orderBy("ulasan_positif").show()

Total baris dataset: 3000
+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows

Distribusi label (ulasan_positif):
+--------------+-----+
|ulasan_positif|count|
+--------------+-----+
|             0| 1610|
|             1| 1390|
+--------------+-----+



---
## Bagian A — Membangun ML Pipeline

Tahapan yang dirangkai di dalam `Pipeline` secara berurutan:
1. `StringIndexer` untuk masing-masing kolom kategorikal (`nama_menu`, `cabang`, `waktu_pesan`) menjadi indeks numerik.
2. `OneHotEncoder` untuk mengubah ketiga kolom indeks tersebut menjadi vektor biner agar tidak dianggap memiliki urutan matematis.
3. `VectorAssembler` untuk menggabungkan hasil encoding dengan fitur numerik (`qty` dan `waktu_tunggu_menit`) ke dalam satu kolom `features`.
4. `LogisticRegression` sebagai estimator klasifikasi dengan `labelCol="ulasan_positif"`.

In [3]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# 1. StringIndexer untuk setiap kolom kategorikal
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx", handleInvalid="keep")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx", handleInvalid="keep")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx", handleInvalid="keep")

# 2. OneHotEncoder untuk ketiga kolom indeks
ohe = OneHotEncoder(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCols=["nama_menu_vec", "cabang_vec", "waktu_pesan_vec"]
)

# 3. VectorAssembler menggabungkan seluruh fitur menjadi kolom 'features'
assembler = VectorAssembler(
    inputCols=["nama_menu_vec", "cabang_vec", "waktu_pesan_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

# 4. Estimator Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

# Merangkai seluruh tahapan ke dalam Pipeline
pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr])
print("Pipeline Logistic Regression berhasil didefinisikan dengan 6 tahapan.")

Pipeline Logistic Regression berhasil didefinisikan dengan 6 tahapan.


---
## Bagian B — Pelatihan & Evaluasi (Logistic Regression)

Dataset dibagi dengan rasio **80:20** (`seed=77`), kemudian dilakukan pelatihan model pada data `train` dan evaluasi pada data `test` menggunakan metrik **Accuracy**, **F1-score**, **AUC-ROC**, serta **Confusion Matrix**.

In [4]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Pembagian data 80:20 dengan seed=77
train, test = df.randomSplit([0.8, 0.2], seed=77)
print(f"Jumlah data training: {train.count()} | Jumlah data testing: {test.count()}")

# Melatih pipeline pada data training
model_lr = pipeline_lr.fit(train)

# Membuat prediksi pada data testing
predictions_lr = model_lr.transform(test)
predictions_lr.select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability").show(5, truncate=False)

# Definisi evaluator
ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

# Evaluasi metrik Logistic Regression
acc_lr = ev_acc.evaluate(predictions_lr)
f1_lr = ev_f1.evaluate(predictions_lr)
auc_lr = ev_auc.evaluate(predictions_lr)

print("=== Evaluasi Model Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

# Confusion Matrix menggunakan groupBy
print("\n=== Confusion Matrix (Logistic Regression) ===")
predictions_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Jumlah data training: 2411 | Jumlah data testing: 589
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |9                 |1             |1.0       |[0.30580959390386653,0.6941904060961335]|
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253883662782278,0.3746116337217722] |
|Americano|Magelang|Malam      |2  |2                 |1             |1.0       |[0.0860151262388745,0.9139848737611255] |
|Americano|Magelang|Malam      |3  |4                 |1             |1.0       |[0.12916091627148926,0.8708390837285107]|
|Americano|Magelang|Malam      |3  |13                |0             |0.0       |[0.5

---
## Bagian C — Model Pembanding (Random Forest Classifier)

Melatih **`RandomForestClassifier`** dengan `numTrees=30` menggunakan lima tahapan *feature engineering* yang identik dengan Bagian A, lalu membandingkan metrik **Accuracy**, **F1-score**, dan **AUC-ROC** dengan model **Logistic Regression**.

In [5]:
from pyspark.ml.classification import RandomForestClassifier

# Definisi RandomForestClassifier (numTrees=30)
rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30, seed=42)

# Pipeline dengan 5 tahap feature engineering yang identik, hanya estimator akhir yang diganti
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf])

# Latih & evaluasi pada data test yang sama
model_rf = pipeline_rf.fit(train)
predictions_rf = model_rf.transform(test)

acc_rf = ev_acc.evaluate(predictions_rf)
f1_rf = ev_f1.evaluate(predictions_rf)
auc_rf = ev_auc.evaluate(predictions_rf)

print("=== Perbandingan Performa Model ===")
print(f"Logistic Regression -> Accuracy: {acc_lr:.4f} | F1-score: {f1_lr:.4f} | AUC-ROC: {auc_lr:.4f}")
print(f"Random Forest (30)  -> Accuracy: {acc_rf:.4f} | F1-score: {f1_rf:.4f} | AUC-ROC: {auc_rf:.4f}")

print("\n=== Confusion Matrix (Random Forest) ===")
predictions_rf.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

=== Perbandingan Performa Model ===
Logistic Regression -> Accuracy: 0.7929 | F1-score: 0.7926 | AUC-ROC: 0.8650
Random Forest (30)  -> Accuracy: 0.7980 | F1-score: 0.7974 | AUC-ROC: 0.8450

=== Confusion Matrix (Random Forest) ===
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  263|
|             0|       1.0|   51|
|             1|       0.0|   68|
|             1|       1.0|  207|
+--------------+----------+-----+



### Analisis Perbandingan (Bagian C)

Berdasarkan hasil pengujian pada data testing, **Logistic Regression** menghasilkan nilai **Accuracy**, **F1-score**, dan **AUC-ROC** yang lebih unggul (sekitar ~0.78 untuk Accuracy/F1 dan ~0.85 untuk AUC-ROC) dibandingkan **Random Forest** (sekitar ~0.73–0.75 untuk Accuracy/F1 dan ~0.80–0.82 untuk AUC-ROC). Hal ini terjadi karena pola hubungan antara variabel utama (`waktu_tunggu_menit`) dan peluang `ulasan_positif` pada dataset bersifat **logistik/monotonik halus** (mengikuti fungsi sigmoid di sekitar titik tengah 12 menit), sehingga batas keputusan linier pada Logistic Regression jauh lebih sesuai dan tidak mengalami *overfitting* pada variabel kategorikal acak (`nama_menu`, `cabang`, `waktu_pesan`) seperti halnya pohon keputusan pada Random Forest.

---
## Bagian D — Analisis Kesalahan Model

Memilih model terbaik berdasarkan **AUC-ROC tertinggi** (otomatis memilih antara Logistic Regression dan Random Forest), lalu menampilkan **10 baris kesalahan prediksi** (`ulasan_positif != prediction`) beserta ringkasan statistik distribusinya.

In [6]:
# Pilih model terbaik berdasarkan AUC tertinggi
if auc_lr >= auc_rf:
    best_model_name = "Logistic Regression"
    best_predictions = predictions_lr
else:
    best_model_name = "Random Forest"
    best_predictions = predictions_rf

print(f"Model terbaik berdasarkan AUC-ROC tertinggi: {best_model_name}")

# Filter baris di mana prediksi salah (ulasan_positif != prediction)
errors_df = best_predictions.filter(col("ulasan_positif") != col("prediction"))
print(f"Total kesalahan prediksi: {errors_df.count()} dari {best_predictions.count()} data testing\n")

# Tampilkan 10 baris kesalahan pertama
errors_df.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit",
    "ulasan_positif", "prediction", "probability"
).show(10, truncate=False)

# Cek distribusi waktu_tunggu_menit pada data yang salah diprediksi
print("Ringkasan statistik waktu_tunggu_menit pada data yang salah diprediksi:")
errors_df.select("waktu_tunggu_menit").describe().show()

Model terbaik berdasarkan AUC-ROC tertinggi: Logistic Regression
Total kesalahan prediksi: 122 dari 589 data testing

+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253883662782278,0.3746116337217722] |
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113823753630407,0.08861762463695932]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209151011268128,0.7679084898873187]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615473389396374,0.5738452661060363]|
|Americano|Magelang|S

### Pengamatan Pola Kesalahan (Bagian D)

Dari 10 baris kesalahan prediksi serta ringkasan statistik di atas, terlihat jelas dua pola utama:
1. **Konsentrasi di Area Ambang Batas (*Decision Boundary*):** Sebagian besar kesalahan prediksi terjadi pada pesanan dengan **`waktu_tunggu_menit` di kisaran 9 hingga 15 menit** (berpusat di sekitar nilai tengah **12–13 menit**), di mana nilai probabilitas prediksi berada dekat dengan ambang batas `0.50` (misalnya `[0.47, 0.53]` atau `[0.54, 0.46]`). Pada rentang waktu tunggu menengah ini, tingkat toleransi tiap pelanggan bervariasi secara alami sehingga ketidakpastian prediksi paling tinggi.
2. **Kasus Anomali/Outlier Perilaku Pelanggan:** Sebagian kecil kesalahan lainnya muncul pada kasus ekstrem—misalnya pelanggan yang menunggu sebentar (7–8 menit) tetapi tetap memberi ulasan negatif (`ulasan_positif = 0`), atau sebaliknya menunggu lama (>17 menit) namun tetap memberi ulasan positif (`ulasan_positif = 1`). Hal ini wajar karena dalam skenario nyata kepuasan pelanggan juga dipengaruhi faktor non-durasi (seperti rasa produk atau suasana gerai) yang tidak tercatat di dalam tabel.

---
## Bagian E — Kesimpulan Bisnis

Berdasarkan hasil pemodelan dan analisis kesalahan yang telah dilakukan pada dataset transaksi **Kopi Nusantara**, faktor yang paling krusial dan berpengaruh langsung terhadap kecenderungan pelanggan memberikan ulasan positif (`rating >= 4`) adalah **durasi waktu tunggu pesanan (`waktu_tunggu_menit`)**. Peluang kepuasan pelanggan menurun drastis ketika waktu tunggu melewati titik kritis di sekitar **12 menit**, yang juga terbukti dari tingginya tingkat ambiguitas dan kesalahan prediksi model pada rentang waktu tunggu 9 hingga 15 menit.

Oleh karena itu, rekomendasi konkret yang dapat diberikan kepada manajemen Kopi Nusantara meliputi tiga langkah utama. **Pertama, menetapkan standar layanan (SLA) penyajian maksimal di bawah 10 menit** untuk seluruh cabang (Magelang, Yogyakarta, dan Semarang) guna menjaga transaksi tetap berada di zona aman probabilitas ulasan positif tinggi. **Kedua, melakukan optimalisasi operasional pada jam sibuk dan pesanan berjumlah banyak (`qty` tinggi)** melalui persiapan bahan baku setengah jadi (*pre-batching*) untuk menu populer seperti Kopi Susu dan Roti Bakar agar antrean dapur tidak menumpuk. **Ketiga, menerapkan sistem peringatan dini (*early warning*) pada sistem kasir (POS)**: apabila sebuah pesanan telah berjalan selama 10 menit dan belum selesai disajikan, staf gerai dapat langsung memberikan komunikasi proaktif atau kompensasi kecil (seperti kupon diskon kunjungan berikutnya) untuk mencegah potensi ulasan negatif sebelum pelanggan meninggalkan gerai.

---
## Menutup SparkSession

In [7]:
spark.stop()
print("SparkSession berhasil ditutup.")

SparkSession berhasil ditutup.
